In [3]:
# ============================================================
# FINAL HEATWAVE ANALYSIS + DAILY HEATWAVE MASK
# South Africa maize season, 1981–2025
# ============================================================

import os
import numpy as np
import xarray as xr


# ============================================================
# 1. FILE PATHS
# ============================================================

input_file = r"C:\Users\franc\era5_daily_max_2m_temperature.nc"

metrics_file = (
    r"C:\Users\franc\south_africa_maize_season_heatwaves_1981_2025_updated.nc"
)

mask_file = (
    r"C:\Users\franc\heatwave_daily_mask_1981_2025.nc"
)


# ============================================================
# 2. CHECK INPUT FILE
# ============================================================

print("=" * 70)
print("HEATWAVE ANALYSIS + DAILY MASK")
print("=" * 70)

print("\nInput file:")
print(input_file)

print("\nFile exists:", os.path.exists(input_file))

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"Temperature file not found:\n{input_file}"
    )


# ============================================================
# 3. OPEN DATASET
# ============================================================

print("\nLoading netCDF dataset...")

ds = xr.open_dataset(input_file)

print("\nVariables:")
print(list(ds.data_vars))

print("\nDimensions:")
print(dict(ds.sizes))


# ============================================================
# 4. SELECT DAILY MAXIMUM TEMPERATURE VARIABLE
# ============================================================

if "t2m_max" in ds.data_vars:

    tmax = ds["t2m_max"]

elif "t2m" in ds.data_vars:

    tmax = ds["t2m"]

else:

    raise ValueError(
        f"No suitable temperature variable found. "
        f"Available variables: {list(ds.data_vars)}"
    )


print("\nUsing temperature variable:")
print(tmax.name)

print("\nShape:")
print(tmax.shape)

print("\nFirst date:")
print(tmax.time.values[0])

print("\nLast date:")
print(tmax.time.values[-1])

print("\nOriginal units:")
print(
    tmax.attrs.get(
        "units",
        "Unknown"
    )
)


# ============================================================
# 5. CHECK FULL TIME SERIES
# ============================================================

expected_days = 16436

print("\nTime check")
print("-" * 40)

print(
    "Actual days:",
    tmax.sizes["time"]
)

print(
    "Expected days:",
    expected_days
)

print(
    "Unique dates:",
    len(
        np.unique(
            tmax.time.values
        )
    )
)

if tmax.sizes["time"] != expected_days:
    raise ValueError(
        "Temperature dataset does not contain "
        "the expected 16436 daily observations."
    )

if (
    len(np.unique(tmax.time.values))
    != tmax.sizes["time"]
):

    raise ValueError(
        "Duplicate temperature dates detected."
    )

print("Time check passed!")


# ============================================================
# 6. CONVERT KELVIN TO CELSIUS
# ============================================================

original_units = tmax.attrs.get(
    "units",
    ""
)

if (
    original_units == "K"
    or float(
        tmax.max(skipna=True).values
    ) > 200
):

    print(
        "\nConverting Kelvin to Celsius..."
    )

    tmax = (
        tmax
        -
        273.15
    )

    tmax.attrs[
        "units"
    ] = "degrees_Celsius"

else:

    print(
        "\nTemperature already appears "
        "to be in Celsius."
    )


print(
    "Final units:",
    tmax.attrs.get(
        "units",
        "Unknown"
    )
)


# ============================================================
# 7. DEFINE MAIZE GROWING SEASON
# ============================================================

growing_months = [
    10,
    11,
    12,
    1,
    2,
    3,
    4
]


growing_mask = (
    tmax.time.dt.month.isin(
        growing_months
    )
)


tmax_growing = tmax.sel(
    time=growing_mask
)


print(
    "\nGrowing-season observations:",
    tmax_growing.sizes["time"]
)


# ============================================================
# 8. CALCULATE DAILY 90TH-PERCENTILE THRESHOLDS
# ============================================================

print(
    "\nCalculating growing-season "
    "90th-percentile thresholds..."
)


thresh = (
    tmax_growing
    .groupby(
        "time.dayofyear"
    )
    .quantile(
        0.90,
        dim="time"
    )
)


print(
    "Threshold calculation complete."
)


# ============================================================
# 9. IDENTIFY DAILY THRESHOLD EXCEEDANCES
# ============================================================

print(
    "\nIdentifying threshold exceedance days..."
)


threshold_growing = thresh.sel(
    dayofyear=
        tmax_growing.time.dt.dayofyear
)


exceeds_growing = (
    tmax_growing
    >
    threshold_growing
)


# ============================================================
# 10. PLACE EXCEEDANCES BACK ON FULL CALENDAR
# ============================================================
#
# Non-growing months remain zero.
#
# This prevents April and October from becoming artificially
# consecutive when applying the 3-day rule.
# ============================================================

binary_full = xr.zeros_like(
    tmax,
    dtype=np.int8
)


binary_full.loc[
    {
        "time":
            tmax_growing.time
    }
] = exceeds_growing.astype(
    np.int8
)


print(
    "Full daily exceedance calendar created."
)


# ============================================================
# 11. FUNCTION TO KEEP RUNS >= 3 CONSECUTIVE DAYS
# ============================================================

def filter_consecutive_days(
    arr,
    min_days=3
):

    arr = np.asarray(
        arr,
        dtype=np.int8
    )

    bounded = np.hstack(
        (
            [0],
            arr,
            [0]
        )
    )

    diffs = np.diff(
        bounded
    )

    run_starts = np.where(
        diffs == 1
    )[0]

    run_ends = np.where(
        diffs == -1
    )[0]

    run_lengths = (
        run_ends
        -
        run_starts
    )

    clean = np.zeros_like(
        arr,
        dtype=np.int8
    )

    for (
        start,
        end,
        length
    ) in zip(
        run_starts,
        run_ends,
        run_lengths
    ):

        if length >= min_days:

            clean[
                start:end
            ] = 1

    return clean


# ============================================================
# 12. CREATE DAILY HEATWAVE MASK
# ============================================================

print(
    "\nApplying >=3 consecutive-day "
    "heatwave criterion..."
)


hw_mask = xr.apply_ufunc(

    filter_consecutive_days,

    binary_full,

    input_core_dims=[
        ["time"]
    ],

    output_core_dims=[
        ["time"]
    ],

    vectorize=True,

    output_dtypes=[
        np.int8
    ]
)


# apply_ufunc may move time to the end.
# Restore standard order.

hw_mask = hw_mask.transpose(
    "time",
    "latitude",
    "longitude"
)


hw_mask.name = (
    "heatwave_active"
)


hw_mask.attrs[
    "long_name"
] = (
    "Daily agricultural heatwave active mask"
)


hw_mask.attrs[
    "description"
] = (
    "1 = daily Tmax exceeds the localized "
    "90th percentile for at least 3 consecutive "
    "days during the South African maize growing "
    "season; 0 = otherwise"
)


# ============================================================
# 13. CHECK DAILY HEATWAVE MASK
# ============================================================

print(
    "\nDaily heatwave mask created."
)

print(
    "Shape:",
    hw_mask.shape
)

print(
    "First date:",
    hw_mask.time.values[0]
)

print(
    "Last date:",
    hw_mask.time.values[-1]
)

print(
    "Minimum:",
    int(
        hw_mask.min().values
    )
)

print(
    "Maximum:",
    int(
        hw_mask.max().values
    )
)

print(
    "Total active pixel-days:",
    int(
        hw_mask.sum().values
    )
)


# ============================================================
# 14. CREATE HARVEST YEAR
# ============================================================

prod_year = (
    tmax.time.dt.year
    +
    xr.where(
        tmax.time.dt.month >= 10,
        1,
        0
    )
)


# ============================================================
# 15. IDENTIFY HEATWAVE EVENT STARTS
# ============================================================

previous_day = hw_mask.shift(
    time=1,
    fill_value=0
)


hw_starts = (
    (hw_mask == 1)
    &
    (previous_day == 0)
).astype(
    np.int8
)


# ============================================================
# 16. RESTRICT ANNUAL METRICS TO GROWING SEASON
# ============================================================

hw_mask_growing = hw_mask.sel(
    time=growing_mask
)


hw_starts_growing = hw_starts.sel(
    time=growing_mask
)


tmax_growing_metrics = tmax.sel(
    time=growing_mask
)


harvest_year_growing = (
    tmax_growing_metrics.time.dt.year
    +
    xr.where(
        tmax_growing_metrics.time.dt.month >= 10,
        1,
        0
    )
)


hw_mask_growing = (
    hw_mask_growing
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year_growing.values
        )
    )
)


hw_starts_growing = (
    hw_starts_growing
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year_growing.values
        )
    )
)


tmax_growing_metrics = (
    tmax_growing_metrics
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year_growing.values
        )
    )
)


# ============================================================
# 17. HEATWAVE FREQUENCY
# ============================================================

print(
    "\nCalculating annual heatwave frequency..."
)


hw_frequency = (
    hw_starts_growing
    .groupby(
        "harvest_year"
    )
    .sum(
        "time"
    )
)


# ============================================================
# 18. HEATWAVE DURATION
# ============================================================

print(
    "Calculating annual heatwave duration..."
)


hw_duration = (
    hw_mask_growing
    .groupby(
        "harvest_year"
    )
    .sum(
        "time"
    )
)


# ============================================================
# 19. HEATWAVE INTENSITY
# ============================================================

print(
    "Calculating annual heatwave intensity..."
)


threshold_broadcasted = thresh.sel(
    dayofyear=
        tmax_growing_metrics.time.dt.dayofyear
)


excess_temp = (
    tmax_growing_metrics
    -
    threshold_broadcasted
).where(
    hw_mask_growing == 1,
    0
)


excess_temp = (
    excess_temp
    .assign_coords(
        harvest_year=(
            "time",
            harvest_year_growing.values
        )
    )
)


hw_intensity = (
    excess_temp
    .groupby(
        "harvest_year"
    )
    .sum(
        "time"
    )
)


# ============================================================
# 20. KEEP COMPLETE HARVEST YEARS
# ============================================================

hw_frequency = (
    hw_frequency
    .sel(
        harvest_year=slice(
            1982,
            2025
        )
    )
)


hw_duration = (
    hw_duration
    .sel(
        harvest_year=slice(
            1982,
            2025
        )
    )
)


hw_intensity = (
    hw_intensity
    .sel(
        harvest_year=slice(
            1982,
            2025
        )
    )
)


# ============================================================
# 21. CREATE ANNUAL METRICS DATASET
# ============================================================

metrics_ds = xr.Dataset(
    {

        "hw_frequency":
            hw_frequency,

        "hw_duration":
            hw_duration,

        "hw_intensity":
            hw_intensity

    }
)


# ============================================================
# 22. CREATE DAILY MASK DATASET
# ============================================================

mask_ds = xr.Dataset(
    {
        "heatwave_active":
            hw_mask
    }
)


# ============================================================
# 23. FINAL ANNUAL METRICS CHECK
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL HEATWAVE METRICS CHECK"
)

print(
    "=" * 70
)


print(
    "Dimensions:",
    dict(
        metrics_ds.sizes
    )
)


print(
    "First harvest year:",
    int(
        metrics_ds
        .harvest_year
        .values[0]
    )
)


print(
    "Last harvest year:",
    int(
        metrics_ds
        .harvest_year
        .values[-1]
    )
)


print(
    "Maximum frequency:",
    int(
        metrics_ds[
            "hw_frequency"
        ].max().values
    )
)


print(
    "Maximum duration:",
    int(
        metrics_ds[
            "hw_duration"
        ].max().values
    )
)


print(
    "Maximum intensity:",
    float(
        metrics_ds[
            "hw_intensity"
        ].max().values
    )
)


# ============================================================
# 24. SAVE ANNUAL METRICS
# ============================================================

print(
    "\nSaving annual heatwave metrics..."
)


metrics_ds.to_netcdf(
    metrics_file
)


print(
    "Saved:"
)

print(
    metrics_file
)


# ============================================================
# 25. SAVE DAILY MASK
# ============================================================

print(
    "\nSaving daily heatwave mask..."
)


mask_ds.to_netcdf(
    mask_file
)


print(
    "Saved:"
)

print(
    mask_file
)


# ============================================================
# 26. FINAL DAILY MASK CHECK
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL DAILY HEATWAVE MASK"
)

print(
    "=" * 70
)


print(
    "Shape:",
    mask_ds[
        "heatwave_active"
    ].shape
)


print(
    "First date:",
    mask_ds[
        "heatwave_active"
    ].time.values[0]
)


print(
    "Last date:",
    mask_ds[
        "heatwave_active"
    ].time.values[-1]
)


print(
    "Minimum:",
    int(
        mask_ds[
            "heatwave_active"
        ].min().values
    )
)


print(
    "Maximum:",
    int(
        mask_ds[
            "heatwave_active"
        ].max().values
    )
)


print(
    "\nDONE!"
)

HEATWAVE ANALYSIS + DAILY MASK

Input file:
C:\Users\franc\era5_daily_max_2m_temperature.nc

File exists: True

Loading netCDF dataset...

Variables:
['t2m']

Dimensions:
{'time': 16436, 'latitude': 57, 'longitude': 89}

Using temperature variable:
t2m

Shape:
(16436, 57, 89)

First date:
1981-01-01T00:00:00.000000000

Last date:
2025-12-31T00:00:00.000000000

Original units:
K

Time check
----------------------------------------
Actual days: 16436
Expected days: 16436
Unique dates: 16436
Time check passed!

Converting Kelvin to Celsius...
Final units: degrees_Celsius

Growing-season observations: 9551

Calculating growing-season 90th-percentile thresholds...
Threshold calculation complete.

Identifying threshold exceedance days...
Full daily exceedance calendar created.

Applying >=3 consecutive-day heatwave criterion...

Daily heatwave mask created.
Shape: (16436, 57, 89)
First date: 1981-01-01T00:00:00.000000000
Last date: 2025-12-31T00:00:00.000000000
Minimum: 0
Maximum: 1
Total ac

In [4]:
# ============================================================
# STAGE 4: ALIGN DAILY MASKS + CREATE CDHW ACTIVE MASK
# ============================================================

import os
import numpy as np
import xarray as xr

# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------

fd_file = r"C:\Users\franc\flash_drought_daily_mask_1981_2025.nc"

hw_file = r"C:\Users\franc\heatwave_daily_mask_1981_2025.nc"

output_file = r"C:\Users\franc\cdhw_daily_mask_1981_2025.nc"


# ------------------------------------------------------------
# 2. OPEN FILES
# ------------------------------------------------------------

fd_ds = xr.open_dataset(fd_file)
hw_ds = xr.open_dataset(hw_file)

fd = fd_ds["flash_drought_active"]
hw = hw_ds["heatwave_active"]


print("=" * 70)
print("CDHW DAILY MASK")
print("=" * 70)

print("\nFlash drought:")
print("Shape:", fd.shape)
print("Dims :", fd.dims)

print("\nHeatwave:")
print("Shape:", hw.shape)
print("Dims :", hw.dims)


# ------------------------------------------------------------
# 3. STANDARDISE COORDINATE NAMES
# ------------------------------------------------------------

fd = fd.rename({
    "lat": "latitude",
    "lon": "longitude"
})


# ------------------------------------------------------------
# 4. CHECK TIME ALIGNMENT
# ------------------------------------------------------------

print("\nTime check")

print(
    "FD:",
    fd.time.values[0],
    "to",
    fd.time.values[-1]
)

print(
    "HW:",
    hw.time.values[0],
    "to",
    hw.time.values[-1]
)

print(
    "Same number of days:",
    fd.sizes["time"] == hw.sizes["time"]
)

print(
    "Time coordinates identical:",
    np.array_equal(
        fd.time.values,
        hw.time.values
    )
)

if not np.array_equal(
    fd.time.values,
    hw.time.values
):
    raise ValueError(
        "Flash drought and heatwave dates do not match."
    )


# ------------------------------------------------------------
# 5. CHECK GRID RANGES
# ------------------------------------------------------------

print("\nFlash-drought grid:")
print(
    float(fd.latitude.min()),
    "to",
    float(fd.latitude.max())
)
print(
    float(fd.longitude.min()),
    "to",
    float(fd.longitude.max())
)

print("\nHeatwave grid:")
print(
    float(hw.latitude.min()),
    "to",
    float(hw.latitude.max())
)
print(
    float(hw.longitude.min()),
    "to",
    float(hw.longitude.max())
)


# ------------------------------------------------------------
# 6. SORT COORDINATES
# ------------------------------------------------------------

fd = fd.sortby("latitude")
fd = fd.sortby("longitude")

hw = hw.sortby("latitude")
hw = hw.sortby("longitude")


# ------------------------------------------------------------
# 7. REGRID FLASH DROUGHT MASK TO HEATWAVE GRID
# ------------------------------------------------------------
#
# Since this is a binary categorical mask, use nearest-neighbour
# interpolation rather than linear interpolation.
# ------------------------------------------------------------

print(
    "\nRegridding flash drought mask "
    "onto heatwave grid..."
)

fd_on_hw_grid = fd.interp(
    latitude=hw.latitude,
    longitude=hw.longitude,
    method="nearest"
)

fd_on_hw_grid = fd_on_hw_grid.astype(
    np.int8
)


print(
    "Regridded FD shape:",
    fd_on_hw_grid.shape
)

print(
    "HW shape:",
    hw.shape
)


# ------------------------------------------------------------
# 8. CHECK BINARY VALUES
# ------------------------------------------------------------

print(
    "\nFD min/max:",
    int(fd_on_hw_grid.min().values),
    int(fd_on_hw_grid.max().values)
)

print(
    "HW min/max:",
    int(hw.min().values),
    int(hw.max().values)
)


# ------------------------------------------------------------
# 9. CREATE SIMULTANEOUS CDHW MASK
# ------------------------------------------------------------

print(
    "\nCreating simultaneous compound mask..."
)

cdhw_active = xr.where(
    (fd_on_hw_grid == 1)
    &
    (hw == 1),
    1,
    0
).astype(
    np.int8
)

cdhw_active.name = "cdhw_active"

cdhw_active.attrs["long_name"] = (
    "Daily simultaneous compound drought-heatwave active mask"
)

cdhw_active.attrs["description"] = (
    "1 = flash drought and heatwave active "
    "simultaneously at the same grid cell and day; "
    "0 = otherwise"
)


# ------------------------------------------------------------
# 10. CHECK RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CDHW MASK CHECK")
print("=" * 70)

print(
    "Shape:",
    cdhw_active.shape
)

print(
    "First date:",
    cdhw_active.time.values[0]
)

print(
    "Last date:",
    cdhw_active.time.values[-1]
)

print(
    "Minimum:",
    int(cdhw_active.min().values)
)

print(
    "Maximum:",
    int(cdhw_active.max().values)
)

print(
    "Total CDHW pixel-days:",
    int(cdhw_active.sum().values)
)


# ------------------------------------------------------------
# 11. TEST LOCATION
# ------------------------------------------------------------

test = cdhw_active.sel(
    latitude=-27.95,
    longitude=21.05,
    method="nearest"
)

print("\nTest location")
print("-" * 40)

print(
    "Latitude:",
    float(test.latitude)
)

print(
    "Longitude:",
    float(test.longitude)
)

print(
    "Total compound days:",
    int(test.sum().values)
)


# ------------------------------------------------------------
# 12. SAVE
# ------------------------------------------------------------

cdhw_active.to_dataset().to_netcdf(
    output_file
)

print("\nSaved successfully:")
print(output_file)

print("\nDONE!")

CDHW DAILY MASK

Flash drought:
Shape: (16436, 141, 220)
Dims : ('time', 'lat', 'lon')

Heatwave:
Shape: (16436, 57, 89)
Dims : ('time', 'latitude', 'longitude')

Time check
FD: 1981-01-01T00:00:00.000000000 to 2025-12-31T00:00:00.000000000
HW: 1981-01-01T00:00:00.000000000 to 2025-12-31T00:00:00.000000000
Same number of days: True
Time coordinates identical: True

Flash-drought grid:
-34.95000000000313 to -20.950000000003925
11.049999999989154 to 32.94999999998791

Heatwave grid:
-35.0 to -21.0
11.0 to 33.0

Regridding flash drought mask onto heatwave grid...


C:\Users\franc\anaconda3\Lib\site-packages\xarray\core\duck_array_ops.py:250: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Regridded FD shape: (16436, 57, 89)
HW shape: (16436, 57, 89)

FD min/max: 0 1
HW min/max: 0 1

Creating simultaneous compound mask...

CDHW MASK CHECK
Shape: (16436, 57, 89)
First date: 1981-01-01T00:00:00.000000000
Last date: 2025-12-31T00:00:00.000000000
Minimum: 0
Maximum: 1
Total CDHW pixel-days: 371850

Test location
----------------------------------------
Latitude: -28.0
Longitude: 21.0
Total compound days: 95

Saved successfully:
C:\Users\franc\cdhw_daily_mask_1981_2025.nc

DONE!
